# BNP / ANP 选择性 (v4)

**用法：先选好 GPU，然后「代码执行程序 → 全部运行」，不用一格一格点。**

v4 和 v3 的科学部分**一模一样**：同样三个靶点、binder 长度 90、
`--num_designs 60 --budget 20 --diffusion_batch_size 10 --use_kernels false --reuse`。

v4 只改了工程部分：

1. **每跑完一个靶点立刻存 Google Drive**，不等三个都跑完
2. **开头自动从 Drive 把上次的进度拿回来**，配合 `--reuse` 接着跑
3. **每一格都能重复跑**：装过的不重装、跑完的靶点自动跳过
4. 某个靶点出错或被中断，循环**停在那里**，不会偷偷跳到下一个（v3 那个坑）

所以运行时被回收、换一台机器，重新「全部运行」一次就行，不会从零开始。

---

### 跑之前只做两件事

1. **代码执行程序 → 更改运行时类型 → 选 A100 或 L4 → 保存**（峰值显存 3.3 GB，L4 够用）
   **选好就别再改。中途改运行时类型会销毁运行时，`/content` 全清空。**
2. **代码执行程序 → 全部运行**

第 1 格会弹窗让你选 Google 账号授权 Drive，**点「允许」**，然后就不用管了。

备份路径：`我的云端硬盘 / binder_selectivity /`
`results/` 是三个 CSV（真正要的东西），`runs/` 是用来接着跑的完整目录。


## 1. 挂 Google Drive

**这一格会弹窗，选你的 Google 账号，点「允许」。** 整个 notebook 只有这一步需要你动手。

In [ ]:
import os, pathlib
from google.colab import drive
if not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive')
else:
    print("Drive 已经挂好了")

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["UV_CACHE_DIR"] = "/content/uv_cache"

DRIVE_ROOT = "/content/drive/MyDrive/binder_selectivity"
DRIVE_RUNS = DRIVE_ROOT + "/runs"
DRIVE_CSV  = DRIVE_ROOT + "/results"
for d in [DRIVE_ROOT, DRIVE_RUNS, DRIVE_CSV]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("备份目录：", DRIVE_ROOT)

## 2. 看一眼硬件

打印显卡型号和剩余磁盘。**确认是你选的那张卡 —— 从这里开始到跑完，别改运行时类型。**

In [ ]:
!nvidia-smi
import shutil; print("可用磁盘 GB: %.0f" % (shutil.disk_usage("/content").free/1e9))

## 3. 装 BoltzGen（装过就跳过，红字无视）

In [ ]:
!pip install -q boltzgen
!boltzgen --version

## 4. 从 Drive 恢复权重缓存（约 6 GB，省下重新下载的时间）

In [ ]:
import pathlib
BK = "/content/drive/MyDrive/boltzgen_cache/hf_cache.tar"
if pathlib.Path("/content/hf_cache").exists() and any(pathlib.Path("/content/hf_cache").iterdir()):
    print("本地已经有权重，跳过")
elif pathlib.Path(BK).exists():
    print("找到权重备份，恢复中…")
    !mkdir -p /content/hf_cache && tar -xf "$BK" -C /content/hf_cache
    print("恢复完成")
else:
    print("没有权重备份，这次会从网上下（约 6 GB）。最后一格会自动存到 Drive，下次就不用下了。")

## 5. 备份 / 恢复工具（只是定义函数，1 秒）

In [ ]:
import os, json, shutil, subprocess, pathlib, datetime

MAX_RUN_GB = 8   # 一个 run 目录超过这个体积，就只备份 final_ranked_designs

def _gb(path):
    out = subprocess.run(["du", "-sb", path], capture_output=True, text=True).stdout.split()
    return int(out[0]) / 1e9 if out else 0.0

def backup_run(local_dir, name, status="complete"):
    """把 /content/<name>_run 存到 Drive。status: complete 跑完了 / partial 半截"""
    local = pathlib.Path(local_dir)
    if not local.exists():
        print("  [备份] 没有 %s，跳过" % local_dir); return
    dst = pathlib.Path(DRIVE_RUNS) / local.name
    dst.mkdir(parents=True, exist_ok=True)

    # 1) 先救 CSV —— 最小、最重要，一秒钟的事
    csv = local / "final_ranked_designs" / "all_designs_metrics.csv"
    if csv.exists():
        shutil.copy2(str(csv), "%s/%s_all_designs_metrics.csv" % (DRIVE_CSV, name))
        print("  [备份] CSV → binder_selectivity/results/%s_all_designs_metrics.csv" % name)

    # 2) 再存整个目录（用来下次 --reuse 接着跑）
    size = _gb(local_dir)
    if size <= MAX_RUN_GB:
        subprocess.run(["rsync", "-a", str(local) + "/", str(dst) + "/"])
        scope = "整个目录 %.1f GB" % size
    else:
        sub = local / "final_ranked_designs"
        if sub.exists():
            subprocess.run(["rsync", "-a", str(sub) + "/", str(dst / "final_ranked_designs") + "/"])
        scope = ("只存了 final_ranked_designs（整个目录 %.1f GB，超过 %d GB 上限，"
                 "中间结构文件没存，换机器接不上，得重跑这个靶点）" % (size, MAX_RUN_GB))

    (dst / "_backup_status.json").write_text(json.dumps(
        {"status": status, "scope": scope,
         "time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}, ensure_ascii=False))
    print("  [备份] %s → Drive，%s，状态 %s" % (local.name, scope, status))

def restore_all():
    """开机时反向操作：把 Drive 里已有的 run 目录拷回 /content"""
    found = []
    for d in sorted(pathlib.Path(DRIVE_RUNS).glob("*")):
        if not d.is_dir():
            continue
        st = {"status": "unknown", "scope": "?", "time": "?"}
        f = d / "_backup_status.json"
        if f.exists():
            try: st = json.loads(f.read_text())
            except Exception: pass
        subprocess.run(["rsync", "-a", "--exclude", "_backup_status.json",
                        str(d) + "/", "/content/" + d.name + "/"])
        found.append((d.name, st))
        print("恢复 %-12s 状态=%-8s 备份于 %s  (%s)" % (d.name, st["status"], st["time"], st["scope"]))
    if not found:
        print("Drive 里还没有 run 目录 —— 第一次跑，正常。")
    else:
        print("\nstatus=complete 的靶点后面会直接跳过；status=partial 的会用 --reuse 接着跑。")
    return found

print("工具就绪")

## 6. 从 Drive 恢复上次的进度

第一次跑会说「还没有 run 目录」，那是对的。

> 万一某个 `partial` 的靶点接着跑时报错，或者最后设计数不对：
> 去 Drive 删掉 `binder_selectivity/runs/<靶点>_run` 整个文件夹，再「全部运行」一次，那个靶点会从头跑。

In [ ]:
restore_all()

## 7. 写三个靶点文件（每次覆盖，内容固定）

In [ ]:
%%writefile /content/bnp.yaml
# 靶点：人 BNP-32（B 型利钠肽，心衰的主要血液标志物）
# 目标：设计一个能结合 BNP 的小蛋白
# 肽序列直接写在这里，不需要 PDB 文件（官方 binding_disordered_peptides 例子就是这么做的）
entities:
  # 要设计的结合蛋白，长度固定 90 残基（固定是为了让三组可比，同时让批处理能用）
  - protein:
      id: A
      sequence: 90

  # 靶点：人 BNP-32，固定不变
  # S P K M V Q G S G C F G R K M D R I S S S S G L G C K V L R R H
  # 1 2 3 4 5 6 7 8 9 10            16        20        26      32
  - protein:
      id: B
      sequence: SPKMVQGSGCFGRKMDRISSSSGLGCKVLRRH

constraints:
  # BNP 的 17 元环：Cys10 - Cys26 之间的二硫键
  - bond:
      atom1: [B, 10, SG]
      atom2: [B, 26, SG]


In [ ]:
%%writefile /content/anp.yaml
# 靶点：人 ANP(1-28)（心房利钠肽）
# 目标：同一套设计流程换成 ANP，用来和 BNP 的结果对比，看选择性
entities:
  # 要设计的结合蛋白，和 BNP 那边同样固定 90 残基
  - protein:
      id: A
      sequence: 90

  # 靶点：人 ANP(1-28)，固定不变
  # S L R R S S C F G G R M D R I G A Q S G L G C N S F R Y
  # 1 2 3 4 5 6 7 8 9 10          15        20  23      28
  - protein:
      id: B
      sequence: SLRRSSCFGGRMDRIGAQSGLGCNSFRY

constraints:
  # ANP 的 17 元环：Cys7 - Cys23 之间的二硫键
  - bond:
      atom1: [B, 7, SG]
      atom2: [B, 23, SG]


In [ ]:
%%writefile /content/scr.yaml
# 阴性对照：把 BNP-32 的序列打乱
# 保留：氨基酸组成完全相同、两个 Cys 仍在第 10 和第 26 位（所以 17 元环结构仍在）
# 改变：序列顺序（与 BNP 仅 3/32 位置相同）
# 用途：如果设计对这条"假靶点"也表现出选择性，说明观察到的 Δ 是噪声
entities:
  - protein:
      id: A
      sequence: 90

  - protein:
      id: B
      sequence: KLMSGGHGSCVGSFRGSKPMRIKDRCSRVLSQ

constraints:
  - bond:
      atom1: [B, 10, SG]
      atom2: [B, 26, SG]


## 8. 验证三个文件（免费，自动判断）

三个都必须是 **Total designed residues: 90**。
不是 90 的话这一格会自己报错停下，后面的 GPU 不会白烧 —— 看到报错就把输出发我。

In [ ]:
import subprocess, re

bad = []
for name in ["bnp", "anp", "scr"]:
    out = subprocess.run(["boltzgen", "check", "/content/%s.yaml" % name],
                         capture_output=True, text=True)
    txt = (out.stdout or "") + (out.stderr or "")
    print("---------- %s ----------" % name)
    print(txt.strip())
    m = re.search(r"Total designed residues:\s*(\d+)", txt)
    if not m or int(m.group(1)) != 90:
        bad.append(name)

if bad:
    raise RuntimeError("这几个靶点的设计残基数不是 90：%s —— 停下，把上面的输出发给 Claude。" % bad)
print("\n三个靶点都是 90 个设计残基，可以跑。")

## 9.（默认跳过）噪声底：同一个靶点跑两遍

11-15 已经测过了，结论记在项目里：ipTM 两批均值差 0.022，单个设计 SD 0.107，
**所以 BNP vs ANP 的 Δ 要明显超过 0.05 才算数。**

「全部运行」会跳过这一节。以后想重测，把下面的 `RUN_NOISE` 改成 `True`。

In [ ]:
RUN_NOISE = False   # 想重测噪声底就改成 True

import time, subprocess
if not RUN_NOISE:
    print("跳过噪声底（RUN_NOISE = False）。已有结论：ipTM 两批均值差 0.022，单设计 SD 0.107，Δ 要超过 0.05。")
else:
    t0 = time.time()
    for tag in ["a", "b"]:
        subprocess.run(["boltzgen", "run", "/content/bnp.yaml",
                        "--output", "/content/noise_%s" % tag,
                        "--protocol", "protein-anything",
                        "--num_designs", "20", "--budget", "20",
                        "--diffusion_batch_size", "10",
                        "--use_kernels", "false", "--reuse"])
        backup_run("/content/noise_%s" % tag, "noise_%s" % tag)
    print("\n两批共用时 %.1f 分钟" % ((time.time() - t0) / 60))

### 9b. 噪声底的数字（没跑就自动跳过，不会卡住后面）

In [ ]:
import pandas as pd, glob
def _load(d):
    f = glob.glob(d + "/final_ranked_designs/all_designs_metrics.csv")
    return pd.read_csv(f[0]) if f else None

a, b = _load("/content/noise_a"), _load("/content/noise_b")
if a is None or b is None:
    print("本次没跑噪声底，跳过。")
else:
    num = a.select_dtypes("number").columns
    cmp = pd.DataFrame({"批A均值": a[num].mean(), "批B均值": b[num].mean(),
                        "批A标准差": a[num].std()})
    cmp["两批均值之差"] = (cmp["批A均值"] - cmp["批B均值"]).abs()
    display(cmp.round(3))
    print("\n某个指标的'两批均值之差'如果和它的'批A标准差'差不多大，说明这个指标在这个样本量下不稳定。")

## 10. 正式批次：三个靶点各 60 个

科学参数和 v3 完全一样。工程上的三点不同：

1. 每个靶点**跑完就立刻存 Drive**
2. 已经有 60 个设计的靶点**自动跳过**
3. 某个靶点报错或被中断，**整个循环停在那里**，不会偷偷跳到下一个

L4 上一个靶点约 35～40 分钟，三个约两小时。中途断了也不亏，重新「全部运行」接着跑。

In [ ]:
import time, glob, subprocess
import pandas as pd

N = 60
targets = ["bnp", "anp", "scr"]

def n_done(name):
    f = glob.glob("/content/%s_run/final_ranked_designs/all_designs_metrics.csv" % name)
    if not f: return 0
    try: return len(pd.read_csv(f[0]))
    except Exception: return 0

for name in targets:
    have = n_done(name)
    if have >= N:
        print("== %s 已经有 %d 个设计，跳过 ==" % (name, have)); continue
    print("\n========== 开始 %s（现在 %d 个，目标 %d 个）==========" % (name, have, N))
    t0 = time.time()
    cmd = ["boltzgen", "run", "/content/%s.yaml" % name,
           "--output", "/content/%s_run" % name,
           "--protocol", "protein-anything",
           "--num_designs", str(N), "--budget", "20",
           "--diffusion_batch_size", "10",
           "--use_kernels", "false", "--reuse"]
    try:
        r = subprocess.run(cmd)
    except KeyboardInterrupt:
        print("\n[被中断] 先把 %s 已经跑出来的部分存到 Drive…" % name)
        backup_run("/content/%s_run" % name, name, status="partial")
        print("存好了。重新「全部运行」就会接着跑。")
        raise
    if r.returncode != 0:
        print("\n[出错] %s 的 boltzgen 返回码 %d。先备份，然后停在这里，不往下跑。" % (name, r.returncode))
        backup_run("/content/%s_run" % name, name, status="partial")
        break
    print("%s 用时 %.1f 分钟" % (name, (time.time() - t0) / 60))
    backup_run("/content/%s_run" % name, name, status="complete")

print("\n本格结束。已跑完：", [t for t in targets if n_done(t) >= N])

## 11. 汇总：每个靶点跑了多少、主要指标、备份在哪

In [ ]:
import glob, pandas as pd, pathlib

RANK = "design_to_target_iptm"   # 排序用的列（跨靶点可比）；quality_score 是批内排名归一化，不能跨靶点比
rows = []
for name in ["bnp", "anp", "scr"]:
    fs = glob.glob("/content/%s_run/final_ranked_designs/all_designs_metrics.csv" % name)
    if not fs:
        rows.append({"靶点": name, "设计数": 0}); continue
    df = pd.read_csv(fs[0])
    r = {"靶点": name, "设计数": len(df)}
    for c in [RANK, "min_design_to_target_pae", "design_to_target_ipsae"]:
        if c in df.columns:
            r[c + "_均值"] = round(df[c].mean(), 3)
    rows.append(r)
display(pd.DataFrame(rows))

print("\nDrive 备份：")
for d in [DRIVE_CSV, DRIVE_RUNS]:
    print(" ", d)
    for p in sorted(pathlib.Path(d).glob("*")):
        print("   -", p.name)

print("\n下一步：把 results/ 里的三个 CSV 下载下来，交给 analysis/cross_score_matrix.py 做交叉打分。")

## 12. 把权重存回 Drive（已经存过就跳过，约 6 GB）

In [ ]:
import pathlib, subprocess
BK = "/content/drive/MyDrive/boltzgen_cache/hf_cache.tar"
pathlib.Path("/content/drive/MyDrive/boltzgen_cache").mkdir(parents=True, exist_ok=True)
if pathlib.Path(BK).exists():
    print("Drive 里已经有权重备份，跳过")
elif pathlib.Path("/content/hf_cache").exists():
    print("打包权重到 Drive，几分钟…")
    subprocess.run(["tar", "-cf", BK, "-C", "/content/hf_cache", "."])
    print("完成")
else:
    print("本地没有 hf_cache，跳过")